<h1 style="color: red; text-align:center">Model Training</br>

---

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from torch.utils.tensorboard import SummaryWriter

import optuna
from tqdm import tqdm

c:\Users\posze\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [3]:
from src.data import build_dataloaders, load_dataloaders_from_preprocessed
from src.feature_extraction_1 import FeatureExtractor
from src.parser import parse_gzip, filter_proteins
from src.encoder import ProteinEncoder

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
train_loader, val_loader, test_loader, mlb, norm_stats = build_dataloaders(
    file_path='../data/uniprot_sprot.dat.gz',
    batch_size=64,
    max_sequence_length=1000,
    num_workers=4,
    max_samples=10000
)

Parsing proteins...
535,892 proteins after length/annotation filter
533,439 proteins | 5,814 GO terms (min_count=50)
Resized to max_samples=10000
Split: train=7,999 | val=1,001 | test=1,000


In [6]:
class BioNet(nn.Module):
    """
    Simple MLP to process the hand-crafted features
    """
    def __init__(self,
                input_dim: int,
                output_dim:int,
                hidden_dims: list = [512, 256],
                dropout: float = 0.3
                ):
        super(BioNet, self).__init__()

        layers = nn.ModuleList()
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, output_dim))

        self.block = nn.Sequential(*layers)

    def forward(self,X: torch.Tensor) -> torch.Tensor:
        return self.block(X)

In [7]:
class ProteinFunctionPredictor(nn.Module):
    """
    Combines BioNet for hand-crafted features and 
    ProteinEncoder for sequence data.
    """

    def __init__(self,
                 physics_dim: int,
                 num_go_terms: int,
                 cnn_channels: int = 64,
                 cnn_depth: int = 3,
                 mlp_hidden_dims: list = [512, 256],
                 physics_out: int = 64,
                 dropout: float = 0.3):
        super(ProteinFunctionPredictor, self).__init__()
        
        self.bionet = BioNet(
            input_dim = physics_dim,
            output_dim = physics_out,
            hidden_dims = mlp_hidden_dims,
            dropout = dropout
        )

        self.protein_encoder = ProteinEncoder(
            in_channels = 20, 
            branch_channels = cnn_channels, 
            kernel_sizes = [3, 5, 9], 
            depth = cnn_depth,
            dropout = dropout
        )

        combined_dim = physics_out + 2 * cnn_channels * 3 # 2 pools per branch [max, mean]

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_go_terms)
        )

    def forward(self,
                physics_features: torch.Tensor,
                sequence_one_hot: torch.Tensor
                ) -> torch.Tensor:
        """
        Args:
            physics_features: Tensor of shape (batch_size, physics_dim)
            sequence_one_hot: Tensor of shape (batch_size, 20, seq_len)

        Returns:
            Tensor of shape (batch_size, num_go_terms) with raw logits
        """
        physics_out = self.bionet(physics_features) # (batch_size, physics_out)
        sequence_out = self.protein_encoder(sequence_one_hot) # (batch_size, cnn_out)

        combined = torch.cat([physics_out, sequence_out], dim=1) # (batch_size, combined_dim)
        logits = self.classifier(combined)
        return logits # (batch_size, num_go_terms)
    
    def predict_proba(self,
                      physics_features: torch.Tensor,
                      sequence_one_hot: torch.Tensor
                      ) -> torch.Tensor:
        """
        Returns predicted probabilities for each GO term
        """
        logits = self.forward(physics_features, sequence_one_hot)
        return torch.sigmoid(logits)



In [8]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    loss_fn: nn.Module,
    max_grad_norm: float = 1.0
) -> float:
    """Trains the model for one epoch and returns the average loss."""

    model.train()
    final_loss = 0.0

    for sequence_one_hot, physics_features, labels in loader:
        sequence_one_hot = sequence_one_hot.to(DEVICE)
        physics_features = physics_features.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(physics_features, sequence_one_hot)
        loss = loss_fn(logits, labels)
        loss.backward()

        # Gradient clpingip
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        optimizer.step()
        final_loss += loss.item() * sequence_one_hot.size(0)
    
    return final_loss / len(loader.dataset)


In [9]:
def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module
) -> float:
    """Evaluates the model on the validation set and returns the average loss."""

    model.eval()
    final_loss = 0.0

    with torch.inference_mode():
        for sequence_one_hot, physics_features, labels in loader:
            sequence_one_hot = sequence_one_hot.to(DEVICE)
            physics_features = physics_features.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(physics_features, sequence_one_hot)
            loss = loss_fn(logits, labels)
            final_loss += loss.item() * sequence_one_hot.size(0)

    return final_loss / len(loader.dataset)

In [10]:
def train_n_epochs(
    fold_id: int,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: optim.Optimizer,
    loss_fn: nn.Module,
    num_epochs: int,
    writer: SummaryWriter,
    checkpoint_path: str,
    write_model: bool = True,
    max_grad_norm: float = 1.0
) -> tuple:
    """
    Trains the model for a specified number of epochs
    """
    train_losses = []
    val_losses = []
    best_val_loss = np.inf

    csv_path = checkpoint_path.replace('.pth', '.csv')

    with open(csv_path, 'w') as f:
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        writer_csv = csv.writer(f)
        writer_csv.writerow(['fold', 'epoch', 'train_loss', 'val_loss'])

    for epoch in tqdm(range(num_epochs), desc=f"Fold {fold_id} Training"):
        train_loss = train_one_epoch(model,train_loader,optimizer,loss_fn,max_grad_norm)
        val_loss = validate_one_epoch(model,val_loader,loss_fn)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        global_step = fold_id * num_epochs + epoch
        writer.add_scalar(f'Loss/Train_fold{fold_id}', train_loss, global_step)
        writer.add_scalar(f'Loss/Val_fold{fold_id}', val_loss, global_step)

        print(f"Fold {fold_id} Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            
            if write_model:
                dic = {
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_losses': train_losses,
                    'val_losses': val_losses,
                }

                os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
                torch.save(dic, checkpoint_path)
    
    return train_losses, val_losses, best_val_loss

In [11]:
def build_best_model(train_loader, val_loader, norm_stats):
    
    kf = KFold(n_splits=5, shuffle=True)

    all_sequence_one_hot = []
    all_physics_features = []
    all_labels = []

    for sequence_one_hot, physics_features, labels in train_loader:
            all_sequence_one_hot.append(sequence_one_hot.cpu())
            all_physics_features.append(physics_features.cpu())
            all_labels.append(labels.cpu())

    for sequence_one_hot, physics_features, labels in val_loader:
            all_sequence_one_hot.append(sequence_one_hot.cpu())
            all_physics_features.append(physics_features.cpu())
            all_labels.append(labels.cpu())

    all_sequence_one_hot = torch.cat(all_sequence_one_hot, dim=0)
    all_physics_features = torch.cat(all_physics_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    def objective(trial: optuna.Trial):
        batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
        lr = trial.suggest_float('lr', 1e-5, 0.01, log=True)
        dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
        cnn_channels = trial.suggest_categorical('cnn_channels', [32, 64, 128])
        cnn_depth = trial.suggest_int('cnn_depth', 2, 4)
        mlp_n_hidden = trial.suggest_int('mlp_n_hidden', 1, 3)
        mlp_hidden_dims = []
        
        for i in range(mlp_n_hidden):
            hidden_dim = trial.suggest_categorical(f'mlp_hidden_dim_{i}', [128, 256, 516])
            mlp_hidden_dims.append(hidden_dim)

        writer = SummaryWriter(log_dir=f'../src/runs/trial_{trial.number}')

        fold_val_losses = np.zeros(kf.n_splits)

        for fold, (train_idx, val_idx) in enumerate(kf.split(all_physics_features)):
            print(f"Fold {fold+1}/{kf.n_splits}")
            fold_train_loader = DataLoader(
                torch.utils.data.TensorDataset(all_sequence_one_hot[train_idx], all_physics_features[train_idx], all_labels[train_idx]),
                batch_size=batch_size,
                shuffle=True
            )
            fold_val_loader = DataLoader(
                torch.utils.data.TensorDataset(all_sequence_one_hot[val_idx], all_physics_features[val_idx], all_labels[val_idx]),
                batch_size=batch_size,
                shuffle=False
            )

            model = ProteinFunctionPredictor(
                physics_dim=norm_stats['physics_dim'],
                num_go_terms=norm_stats['num_go_terms'],
                cnn_channels=cnn_channels,
                cnn_depth=cnn_depth,
                mlp_hidden_dims=mlp_hidden_dims,
                dropout=dropout
            ).to(DEVICE)

            loss_fn = nn.BCEWithLogitsLoss()
            optimizer = optim.Adam(model.parameters(), lr=lr)
            num_epochs = 10

            checkpoint_path = f"../src/models/trial_{trial.number}/fold_{fold}.pth"

            train_losses, val_losses, best_val_loss = train_n_epochs(
                fold_id=fold,
                model=model,
                train_loader=fold_train_loader,
                val_loader=fold_val_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                num_epochs=num_epochs,
                writer=writer,
                checkpoint_path=checkpoint_path,
                write_model=True
            )

            fold_val_losses[fold] = best_val_loss

            trial.report(best_val_loss, fold)

            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            
        mean_val_loss = fold_val_losses.mean()
        
        writer.add_scalar('Optuna/Mean_Val_Loss', mean_val_loss, trial.number)
        writer.close()

        return mean_val_loss
    
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=20)

    # Select 3 best trials
    best_trials = sorted(study.trials, key=lambda t: t.value)[:3]

    return best_trials, study.best_params
   

In [12]:
best_trials, best_params = build_best_model(train_loader, val_loader, norm_stats)
print("Best Hyperparameters:", best_params)

[I 2026-05-02 19:42:14,890] A new study created in memory with name: no-name-2062795c-d9b8-457a-b93f-1d27b7db9135


Fold 1/5


Fold 0 Training:  10%|█         | 1/10 [00:30<04:36, 30.74s/it]

Fold 0 Epoch 1/10 - Train Loss: 0.1668, Val Loss: 0.0071


Fold 0 Training:  20%|██        | 2/10 [00:59<03:58, 29.82s/it]

Fold 0 Epoch 2/10 - Train Loss: 0.0090, Val Loss: 0.0068


... (26 outputs left out)

Fold 1 Training:  60%|██████    | 6/10 [02:55<01:57, 29.27s/it]

Fold 1 Epoch 6/10 - Train Loss: 0.0074, Val Loss: 0.0068


Fold 1 Training:  60%|██████    | 6/10 [03:10<02:07, 31.83s/it]
[W 2026-05-02 19:50:22,193] Trial 0 failed with parameters: {'batch_size': 128, 'lr': 0.0003655930043679468, 'dropout': 0.5, 'cnn_channels': 128, 'cnn_depth': 2, 'mlp_n_hidden': 2, 'mlp_hidden_dim_0': 128, 'mlp_hidden_dim_1': 516} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\posze\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\posze\AppData\Local\Temp\ipykernel_7396\2672013537.py", line 68, in objective
    train_losses, val_losses, best_val_loss = train_n_epochs(
                                              ^^^^^^^^^^^^^^^
  File "C:\Users\posze\AppData\Local\Temp\ipykernel_7396\3996434471.py", line 29, in train_n_epochs
    train_loss = train_one_epoch(model,train_loader,optimizer,loss_fn,max_grad_norm)
            

KeyboardInterrupt: 

In [ ]:
models = {trial.number: [] for trial in best_trials}

for trial in best_trials:
    trial_number = trial.number

    fold_models = []

    for fold in range(5):
        checkpoint_path = f"../src/models/trial_{trial_number}/fold_{fold}.pth"
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

        model = ProteinFunctionPredictor(
            physics_dim=norm_stats['physics_dim'],
            num_go_terms=norm_stats['num_go_terms'],
            cnn_channels=best_params['cnn_channels'],
            cnn_depth=best_params['cnn_depth'],
            mlp_hidden_dims=[best_params[f'mlp_hidden_dim_{i}'] for i in range(best_params['mlp_n_hidden'])],
            dropout=best_params['dropout']
        ).to(DEVICE)

        model.load_state_dict(checkpoint['model_state_dict'])
        fold_models.append(model)

    models[trial.number] = fold_models



---

## Testowanie modelu

---

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, accuracy_score

In [ ]:
def ensamble(models: list, test_loader: DataLoader, threshold: float = 0.5) -> dict:
    """
    Evaluates the ensemble of models on the test set.
    """

    for model in models:
        model.eval()

    all_model_probs = []
    true_labels = []

    with torch.inference_mode():
        for sequence_one_hot, physics_features, labels in test_loader:
            sequence_one_hot = sequence_one_hot.to(DEVICE)
            physics_features = physics_features.to(DEVICE)
            labels = labels.to(DEVICE)

            batch_probs = []
            for model in models:
                probs = model.predict_proba(physics_features, sequence_one_hot)
                batch_probs.append(probs.cpu().numpy())
            
            all_model_probs.append(np.stack(batch_probs))
            true_labels.append(labels.cpu().numpy())
            
    # Concatenate with batches
    # M - models, B - batch size, C - num_GO_Terms, N - total samples
    # (M, B, C) -> (M, N, C)
    all_probs = np.concatenate(all_model_probs, axis=1) # (M, N, C)
    true_labels = np.concatenate(true_labels, axis=0) # (N, C)

    proba_mean = all_probs.mean(axis=0) # (N,C)
    proba_std = all_probs.std(axis=0) # (N,C)
    predictions = (proba_mean >= threshold).astype(int) # (N,C)

    return {
        "predictions": predictions,
        "probabilities": proba_mean,
        "std_deviation": proba_std,
        "labels": true_labels
    }


In [ ]:
ensamble_results = {}

for trial_num, models in models.items():
    print("--------------")
    print(f"Evaluating ensemble for Trial {trial_num}")
    results = ensamble(models, test_loader, threshold=0.5)
    ensamble_results[trial_num] = results

    print(f"Mean std: {results['std_deviation'].mean():.4f}")
    print("--------------")

---

In [ ]:
def calculate_metrics(
        predictions: np.ndarray,
        true_labels: np.ndarray,
        proba_mean: np.ndarray
) -> dict:
    """
    Calculates evaluation metrics for the ensemble predictions.
    """
    metrics = {}
    metrics['accuracy'] = accuracy_score(true_labels.flatten(), predictions.flatten())
    metrics['f1_score'] = f1_score(true_labels.flatten(), predictions.flatten(), average='weighted')
    metrics['precision'] = precision_score(true_labels.flatten(), predictions.flatten(), average='weighted')
    metrics['recall'] = recall_score(true_labels.flatten(), predictions.flatten(), average='weighted')

    # For ROC AUC, we need to calculate it for each class and then average
    num_classes = true_labels.shape[1]
    roc_auc_scores = []
    
    for i in range(num_classes):
        try:
            roc_auc = roc_auc_score(true_labels[:, i], proba_mean[:, i])
            roc_auc_scores.append(roc_auc)
        except ValueError:
            # This can happen if a class is not present in the true labels
            continue
    
    metrics['roc_auc'] = np.mean(roc_auc_scores) if roc_auc_scores else float('nan')

    return metrics

In [ ]:
metrics_all = {}

for trial_num, results in ensamble_results.items():
    metrics = calculate_metrics(
        predictions=results['predictions'],
        true_labels=results['labels'],
        proba_mean=results['probabilities']
    )
    metrics_all[trial_num] = metrics

metrics_df = pd.DataFrame.from_dict(metrics_all, orient='index')
metrics_df.index.name = 'Trial'

print(metrics_df)

In [ ]:
for trial_num, result in ensamble_results.items():
    per_sample = result['proba_std'].mean(axis=1)

    print(f"Trial {trial_num} - Mean per-sample std deviation: {per_sample.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Wykres 1: metryki per ensemble
metric_cols = ['f1_micro', 'f1_macro', 'precision_micro', 'recall_micro', 'roc_auc_micro']
x = np.arange(len(metric_cols))
width = 0.25
for i, (trial_num, m) in enumerate(metrics_all.items()):
    axes[0].bar(x + i * width, [m[c] for c in metric_cols], width, label=f'Trial {trial_num}')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metric_cols, rotation=20, ha='right')
axes[0].set_ylim(0, 1)
axes[0].set_title('Metryki ensembli')
axes[0].legend()

# Wykres 2: rozklad niepewnosci
for trial_num, result in ensamble_results.items():
    unc = result['proba_std'].mean(axis=1)
    axes[1].hist(unc, bins=50, alpha=0.5, label=f'Trial {trial_num}')
axes[1].set_xlabel('Srednia std (niepewnosc per probka)')
axes[1].set_ylabel('Liczba probek')
axes[1].set_title('Rozklad niepewnosci predykcji')
axes[1].legend()

plt.tight_layout()
plt.savefig('../runs/metrics_summary.png', dpi=150)
plt.show()

---

## Statistical Testing

---